# Lesson 10: From a session to an experiment to a benchmark

This walkthrough builds one study step by step. We ask a pretrained SqueezeNet whether
an image shows an animal or a vehicle, record its pooled visual features as an
illustrative `IT` region, and check its answers against held-out CIFAR-10 labels. The
`IT` mapping is a teaching hypothesis, not a validated model of IT.

First the study runs once, by hand: a subject, a stimulus set, a task and one session.
Then the same study is packaged twice. An **experiment** adds named conditions,
repetitions, a check of the whole setup before anything runs, and a saved record. A
**benchmark** adds reference data and a score.

Everything is bundled: no downloads, CPU only, under two minutes. In a notebook, run
from the lesson folder; for this guide, start Python from the `unified/` checkout.

| Step | What is introduced | What it does |
| --- | --- | --- |
| 1. Choose a model | the subject | Declares what it takes in, what it can produce, and its traits. |
| 2. Choose the stimuli | `StimulusSet` | One row per trial, with IDs and input columns. |
| 3. Describe the task | `TaskContext` | What to do, the form of the answer, how a readout is fitted. |
| 4. Run it once | `BufferedSession` | Shows the stimuli once and collects what was asked for. |
| 5. Read the results back | assemblies | The answers and the activity behind them. |
| 6. Make it an experiment | `TaskProtocol`, recorders, `Experiment` | Conditions, repetitions, checks and a saved record. |
| 7. Make it a benchmark | `Benchmark`, reference data, metric | The same procedure, returning a `Score`. |

## Setup

Imports, quiet output, the bundled files and image preprocessing.

In [1]:
# Keep the output short: no progress bars, no library warnings.
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
import tqdm as _tqdm
for _bar in (tqdm, _tqdm.tqdm):
    _bar.__init__ = partialmethod(_bar.__init__, disable=True)
warnings.filterwarnings('ignore')
import os
os.environ['BS_INSTALL_DEPENDENCIES'] = 'no'      # every requirement is already installed

from pathlib import Path
from unittest.mock import patch
import time
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image

started = time.perf_counter()
torch.set_num_threads(1)                   # one CPU thread, so runs repeat exactly
# The bundled files, found from the lesson folder (notebook) or the unified/ checkout (guide).
ASSETS = next(path for path in (Path('../assets/lessons'), Path('notebooks/assets/lessons'))
              if (path / 'squeezenet1_1.pth').is_file())

# The network expects 224 x 224 images, normalized the way it was trained.
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def preprocess(paths):
    """Image files -> one (batch, 3, 224, 224) array."""
    images = []
    for path in paths:
        with Image.open(path) as image:
            images.append(transform(image.convert('RGB')).numpy())
    return np.stack(images)

## The subject

### 1. Choose a model

For a registered model, start with `model = brainscore.load_model('your-registered-id')`
and look at what it declares; its registration already supplies its extractors, readout
and region map.

Here we build a SqueezeNet subject locally so the example runs offline. Pooling gives a
small 512-unit recording, and the readout answers from the same layer.

In [2]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

def make_subject(backbone, identifier):
    # Pool each of the 512 feature maps to one number, so a recording has 512 units.
    network = torch.nn.Sequential(
        backbone.features, torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten()).eval()
    # Build the extractor on the CPU even when a GPU is present, so every machine gets the same numbers.
    with patch('torch.cuda.is_available', return_value=False), \
            patch('torch.backends.mps.is_available', return_value=False):
        extractor = VisionWrapper(model=network, preprocessing=preprocess,
                                  identifier=identifier, kind='frame', batch_size=16)
    # Layer '2' is the Flatten step: it stands for IT and the readout answers from it.
    return BrainScoreModel(identifier=identifier, model=network,
                           extractor={'vision': extractor}, region_layer_map={'IT': '2'},
                           behavioral_readout_layer='2')

backbone = torchvision.models.squeezenet1_1(weights=None)       # weights=None: no download
backbone.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth',
                                    map_location='cpu', weights_only=True))
model = make_subject(backbone, 'lesson-squeezenet1_1-pooled')
print('inputs:', sorted(model.in_channels))
print('outputs:', sorted(model.out_channels))
print('traits:', sorted(model.traits))

inputs: ['vision', 'vision:still']
outputs: ['behavior', 'behavior:choice', 'behavior:probabilities', 'behavior:rating', 'behavior:rt', 'neural:IT']
traits: []


## The experiment

### 2. Choose the stimuli

48 images from four categories of the bundled collection: the first eight of each
category teach the readout, and four other images of each are the test trials. The test
table carries no labels; they become reference data only at step 7.

In [3]:
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

metadata = pd.read_csv(ASSETS / 'images.csv')
metadata = metadata[metadata['category'].isin(['cat', 'dog', 'airplane', 'ship'])].copy()
metadata['stimulus_id'] = metadata['file'].str.removesuffix('.png')
metadata['image_path'] = metadata['file'].map(lambda name: str(ASSETS / 'images' / name))
is_fitting = metadata.groupby('category').cumcount() < 8      # the first 8 of each category

def stimulus_table(rows, identifier, fitting=False):
    # Only the fitting table carries labels; the test table gets IDs and paths only.
    columns = ['stimulus_id', 'image_path'] + (['kind'] if fitting else [])
    table = StimulusSet(rows[columns].rename(columns={'kind': 'label'}).reset_index(drop=True))
    table.identifier = identifier
    table.stimulus_paths = dict(zip(table['stimulus_id'], table['image_path']))
    return table

fitting_stimuli = stimulus_table(metadata[is_fitting], 'lesson-fit-32', fitting=True)
stimuli = stimulus_table(metadata[~is_fitting], 'lesson-test-16')
assert set(fitting_stimuli['stimulus_id']).isdisjoint(stimuli['stimulus_id'])
print(f'{len(fitting_stimuli)} fitting images; {len(stimuli)} test images')

32 fitting images; 16 test images


### 3. Describe the task

`Choice` asks for an animal or vehicle answer and for the implied probability of each.
`fitting_stimuli` are the labelled images the readout learns from, and
`prefer_path='readout'` says to answer through that readout: an instruction does not make
an image network understand language.

In [4]:
from brainscore_core import TaskContext
from brainscore_core.responses import Choice

task = TaskContext(response=Choice(['animal', 'vehicle'], probabilities=True),
                   fitting_stimuli=fitting_stimuli, prefer_path='readout')

## Run

### 4. Run it once

A session is one visit by the subject. This one shows the test images once and asks for
three things: the answers, their probabilities, and the `IT` activity. All three come
from the same presentation of each image. The readout is fitted first, on the fitting
images, in a separate pass.

In [5]:
from brainscore_core.streaming import BufferedSession

# One session: show the test images once; ask for the answers and the IT activity behind them.
session = BufferedSession(stimulus_set=stimuli, task_context=task,
                          requested_output_channels=['behavior:choice', 'behavior:probabilities',
                                                     'neural:IT'])
model.participate(session)
model.reset()                              # forget the fitted readout before anything else runs

## Read results back

### 5. Read the results back

Each measurement is an assembly: one row per test image, matched to its stimulus by
`stimulus_id`. Each recording says how it was made (`attrs['recording']`).

In [6]:
choices = session.collect('behavior:choice')
probabilities = session.collect('behavior:probabilities')
it = session.collect('neural:IT')
assert it.sizes['presentation'] == choices.sizes['presentation'] == len(stimuli)
assert it.sizes['neuroid'] == 512
print('neural:', it.shape, 'choices:', choices.shape)
print('recording:', it.attrs['recording'])
# The chosen label is the column holding the 1.
predicted_labels = choices['choice'].values[choices.values.argmax(axis=1)]
print(pd.DataFrame({'stimulus_id': choices['stimulus_id'].values,
                    'answer': predicted_labels}).head())

neural: (16, 512) choices: (16, 2)
recording: {'task_type': 'choice', 'path': 'readout', 'response': 'Choice', 'forward_pass': 'readout extraction', 'readout_fit': 'current'}
   stimulus_id   answer
0  airplane_08  vehicle
1  airplane_09  vehicle
2  airplane_10  vehicle
3  airplane_11  vehicle
4       cat_08   animal


## Analysis

The reference data are the held-out labels, stored in a `BehavioralAssembly`. They are
CIFAR-10 annotations, not human or neural recordings, so accuracy against them is an
engineering demonstration, not a brain-alignment result. The metric matches answers to
labels by stimulus ID and counts invalid answers as wrong.

In [7]:
from brainscore_core import Score
from brainscore_core.supported_data_standards.brainio.assemblies import BehavioralAssembly

# The held-out labels, one per test image, kept apart from the stimuli the subject saw.
reference = BehavioralAssembly(
    metadata.loc[~is_fitting, 'kind'].to_numpy(), dims=['presentation'],
    coords={'stimulus_id': ('presentation', metadata.loc[~is_fitting, 'stimulus_id'].to_numpy()),
            'category': ('presentation', metadata.loc[~is_fitting, 'category'].to_numpy())})

def accuracy(prediction, reference):
    # Look up each answered image's true label by its stimulus ID, not by row position.
    by_id = dict(zip(reference['stimulus_id'].values, reference.values))
    target = np.array([by_id[identifier] for identifier in prediction['stimulus_id'].values])
    labels = prediction['choice'].values[prediction.values.argmax(axis=1)]
    valid = (prediction['answer_valid'].values if 'answer_valid' in prediction.coords
             else np.ones(len(labels), dtype=bool))
    return Score(float(np.mean((labels == target) & valid)))

## Results

In [8]:
hand_score = accuracy(choices, reference)
print('accuracy:', float(hand_score))

accuracy: 1.0


## Make it an experiment

### 6. Package the study

Running a session by hand is enough to explore. A study needs more, and an `Experiment`
provides it from three pieces:

- **`TaskProtocol`** combines the stimuli and the task under named conditions and
  repetitions. It builds each session itself and resets the subject before and after.
- **Recorders** say what to request and save. `RecordBehavior()` saves every answer the
  task asks for; `RecordNeural(['IT'])` saves the activity of a region the subject declares.
- **`Experiment`** checks the subject, protocol and recorders against each other before
  running (`validate()`), runs the protocol, and writes every recording to `output_dir`.
  That folder must be new, so a run never overwrites an earlier one; the results are read
  back from it, and it can be inspected or replayed later.

The answers it saves are the same as the ones collected by hand.

In [9]:
from tempfile import TemporaryDirectory
from brainscore.experiments import Experiment, RecordBehavior, RecordNeural, TaskProtocol

# The same stimuli and task, under one named condition and one repetition.
protocol = TaskProtocol(stimuli, task, identifier='animal-or-vehicle',
                        conditions=['baseline'], repetitions=[0])

def recorders():
    """What to request and save: the IT activity, and every answer the task asks for."""
    return [RecordNeural(['IT']), RecordBehavior()]

workspace = TemporaryDirectory(prefix='umi-lesson-')   # a folder for this lesson's runs
run_root = Path(workspace.name)
experiment = Experiment(subject=model, protocol=protocol, tools=recorders(),
                        output_dir=run_root / 'exploration')
plan = experiment.validate()               # checks the whole setup without running the model
print(plan['protocol']['config']['compatibility'])
result = experiment.run()
assert result.manifest['status'] == 'complete'

# Read back from the saved files: the same answers and activity as the session run by hand.
saved_choices = result.behavior(condition='baseline', repetition=0, measure='choice')
saved_it = result.neural('IT', condition='baseline', repetition=0)
assert np.array_equal(saved_choices.values, choices.values)
assert np.allclose(saved_it.values, it.values)
print('saved:', sorted(path.name for path in result.directory.iterdir()))   # what the run wrote

{'match': 'full', 'consumed_inputs': ['vision'], 'dropped_inputs': [], 'requested_outputs': ['behavior:choice', 'behavior:probabilities', 'neural:IT']}


saved: ['arrays', 'behavior', 'experiment.json', 'neural', 'result.json']


Four deliberate mistakes, each caught before the model runs. Each prints its reason and
leaves no folder behind.

In [10]:
text_only = StimulusSet({'stimulus_id': ['word'], 'sentence': ['cat']})
text_only.identifier = 'wrong-input-example'
mistakes = [                                      # (name, protocol, tools), each wrong in one way
    ('missing-region', TaskProtocol(stimuli, task), [RecordNeural(['not-a-region'])]),
    ('passive-behavior', TaskProtocol(stimuli, TaskContext(task_type='passive')),
     [RecordBehavior()]),
    ('missing-memory', TaskProtocol(stimuli, task, required_traits={'memory:session'}),
     [RecordBehavior()]),
    ('wrong-input', TaskProtocol(text_only, task), [RecordBehavior()]),
]
for name, wrong_protocol, wrong_tools in mistakes:
    destination = run_root / name
    try:
        Experiment(subject=model, protocol=wrong_protocol, tools=wrong_tools,
                   output_dir=destination).validate()
    except ValueError as error:
        print(f'{name}: {error}')
    else:
        raise AssertionError(f'{name} should have failed validation')
    assert not destination.exists()               # nothing was run or written

missing-region: Subject 'lesson-squeezenet1_1-pooled' is incompatible with protocol 'task': cannot produce requested outputs ['neural:not-a-region']. Subject in_channels=['vision', 'vision:still'], out_channels=['behavior', 'behavior:choice', 'behavior:probabilities', 'behavior:rating', 'behavior:rt', 'neural:IT'].
passive-behavior: A passive task produces no behavior; remove RecordBehavior or give the task a response (Choice, Scale, Text).
missing-memory: This task needs subject traits ['memory:session']; 'lesson-squeezenet1_1-pooled' declares none.
wrong-input: Subject 'lesson-squeezenet1_1-pooled' is incompatible with protocol 'task': takes in none of the protocol inputs ['text']. Subject in_channels=['vision', 'vision:still'], out_channels=['behavior', 'behavior:choice', 'behavior:probabilities', 'behavior:rating', 'behavior:rt', 'neural:IT'].


## Make it a benchmark

### 7. Add reference data and a score

A **benchmark is a protocol plus reference data and a score**. A small `Benchmark`
subclass runs the same experiment in `__call__` and applies the metric. Core has no
constructor that turns a `TaskProtocol` into a benchmark; this is the supported path.

A random-weight network is scored first, as the control.

In [11]:
from brainscore_core import Benchmark

class AnimalVehicleBenchmark(Benchmark):
    identifier = 'lesson-animal-vehicle-accuracy'
    version = 1
    uses_session = True

    def __init__(self, protocol, reference, metric, output_root):
        self.protocol, self.reference, self.metric = protocol, reference, metric
        self.output_root = output_root
        # Resolve the same recorders before copying benchmark declarations.
        protocol.accept_requests(set().union(*(tool.output_channels for tool in recorders())))
        self.required_input_channels = protocol.input_channels
        self.requested_output_channels = protocol.output_channels
        self.required_traits = protocol.required_traits

    def __call__(self, subject):
        from uuid import uuid4
        # Each call runs the same experiment into a new folder, then applies the metric.
        run = Experiment(subject=subject, protocol=self.protocol, tools=recorders(),
                         output_dir=self.output_root / uuid4().hex).run()
        prediction = run.behavior(condition='baseline', repetition=0, measure='choice')
        score = self.metric(prediction, self.reference)
        score.attrs.update(reference='CIFAR-10 labels', metric='accuracy', normalization='raw',
                           experiment_directory=str(run.directory))
        return score

benchmark = AnimalVehicleBenchmark(protocol, reference, accuracy, run_root / 'benchmarks')
assert benchmark.protocol is protocol

In [12]:
import brainscore

torch.manual_seed(7)                       # fixes the random weights of the control
control = make_subject(torchvision.models.squeezenet1_1(weights=None), 'lesson-random-squeezenet')
control_score = brainscore.score(control, benchmark, check_mem=False)
model_score = brainscore.score(model, benchmark, check_mem=False)
# Repackaging the procedure must not change its score.
assert float(model_score) == float(hand_score)
assert float(model_score) == float(accuracy(choices, reference.isel(presentation=slice(None, None, -1))))
print({'random-weight accuracy': float(control_score), 'pretrained accuracy': float(model_score)})
print(model_score.attrs['compatibility'])
print(f'Elapsed: {time.perf_counter() - started:.1f} seconds')

{'random-weight accuracy': 0.75, 'pretrained accuracy': 1.0}
{'match': 'full', 'consumed_inputs': ['vision'], 'dropped_inputs': [], 'requested_outputs': ['behavior:choice', 'behavior:probabilities', 'neural:IT']}
Elapsed: 4.7 seconds


`brainscore.score` checks compatibility before running the benchmark and records the match
on the score. A `CompatibilityError` means the pair is not applicable, not a score of zero.
`check_mem=False` skips the memory probe, which this small example does not need.

In [13]:
workspace.cleanup()

## Summary

- A subject, a stimulus set, a task and one session are enough to run a study once.
- An `Experiment` adds conditions, repetitions, pre-flight checks and a saved record, and gives the same answers.
- A benchmark is the same procedure plus reference data and a score.

**Next:** [Lesson 11](11_interventions_in_an_experiment.ipynb) adds interventions: conditions that change the subject while the same recorders measure it.